# Module 10 — Vector Search and Local Vector Databases

Companion notebook to [`docs/modules/10_vector_search_and_local_vector_databases.md`](../docs/modules/10_vector_search_and_local_vector_databases.md).

Runs for real against all three backends — `NumpyVectorStore` (Module 9's `NumpyEmbeddingIndex`),
`ChromaVectorStore`, and `LanceDBVectorStore`. Chroma and LanceDB are vector database libraries,
not LLM runtimes or model weights, so they're installed on this development machine
(`uv add chromadb lancedb`) and every cell below is a real measurement, not a mock or a
honest-skip. Embeddings come from Module 9's `FakeEmbedder` (genuine bag-of-words hashing),
for the same reason Module 9 used it.


In [1]:
import sys
import tempfile
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_10"))
print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/bhakti/workspace/local_ai_app


## 1. One VectorStore protocol, three real backends — Labs 1-2

In [2]:
from local_ai_rag.embeddings.fake import FakeEmbedder
from local_ai_rag.stores.chroma_store import ChromaVectorStore
from local_ai_rag.stores.lancedb_store import LanceDBVectorStore
from local_ai_rag.stores.numpy_store import NumpyVectorStore

tmp_dir = tempfile.mkdtemp(prefix="module10-nb-")
embedder = FakeEmbedder(dimensions=32)

stores = {
    "numpy": NumpyVectorStore(),
    "chroma": ChromaVectorStore("notebook-demo", path=f"{tmp_dir}/chroma"),
    "lancedb": LanceDBVectorStore("notebook-demo", path=f"{tmp_dir}/lancedb", dimensions=embedder.dimensions),
}
print("Backends:", list(stores.keys()))


Backends: ['numpy', 'chroma', 'lancedb']


## 2. Populate all three stores with the same corpus

In [3]:
from store_comparison import CORPUS, populate

vectors = await populate(stores, embedder)
for name, store in stores.items():
    print(f"{name}: {await store.count()} documents")


numpy: 5 documents
chroma: 5 documents
lancedb: 5 documents


## 3. Brute-force vs. ANN: do all three backends agree on top-k? — Lab 5 setup

In [4]:
query_vector = await embedder.embed_query("I forgot my password")

for name, store in stores.items():
    results = await store.search(query_vector, k=3)
    print(f"{name}:")
    for r in results:
        print(f"  {r.score:.3f}  {r.doc_id:15s}  {r.text}")


numpy:
  0.463  doc_password2    Forgot password recovery steps for your account
  0.183  doc_password     How to reset your password
  0.129  doc_order_code   Your order ACC88213 has shipped and is on its way
chroma:
  0.463  doc_password2    Forgot password recovery steps for your account
  0.183  doc_password     How to reset your password
  0.129  doc_order_code   Your order ACC88213 has shipped and is on its way
lancedb:
  0.463  doc_password2    Forgot password recovery steps for your account
  0.183  doc_password     How to reset your password
  0.129  doc_order_code   Your order ACC88213 has shipped and is on its way


## 4. Metadata filters — Lab 3, pushed down three different ways

In [5]:
for name, store in stores.items():
    results = await store.search(query_vector, k=5, metadata_filter={"category": "account"})
    print(f"{name}: {[r.doc_id for r in results]}")


numpy: ['doc_password2', 'doc_password']
chroma: ['doc_password2', 'doc_password']
lancedb: ['doc_password2', 'doc_password']


## 5. Hybrid search recovers what vector search alone misses — Lab 4

`doc_order_code` shares the exact term "ACC88213" with the query, but its embedding (via
`FakeEmbedder`'s hashing) isn't the closest vector match. Vector-only search ranks it low;
hybrid search (Reciprocal Rank Fusion of vector + keyword rankings) recovers it.

In [6]:
from local_ai_rag.stores.hybrid import hybrid_search

documents = {doc_id: text for doc_id, (text, _metadata) in CORPUS.items()}
code_query_vector = await embedder.embed_query("ACC88213")

vector_only = await stores["numpy"].search(code_query_vector, k=1)
print("vector-only top result:", vector_only[0].doc_id if vector_only else None)

hybrid_results = await hybrid_search(stores["numpy"], documents, query="ACC88213", query_embedding=code_query_vector, k=3)
print("hybrid_search results: ", [r.doc_id for r in hybrid_results])


vector-only top result: doc_password2
hybrid_search results:  ['doc_order_code', 'doc_password2', 'doc_password']


## 6. Incremental updates and deletes — real upsert semantics

In [7]:
for name, store in stores.items():
    before = await store.count()
    await store.add("doc_password", "REVISED: how to reset your password", (await embedder.embed_documents(["REVISED: how to reset your password"]))[0])
    after_update = await store.count()
    await store.delete("doc_shipping")
    after_delete = await store.count()
    print(f"{name}: count before={before}, after upsert={after_update} (should be unchanged), after delete={after_delete}")


numpy: count before=5, after upsert=5 (should be unchanged), after delete=4
chroma: count before=5, after upsert=5 (should be unchanged), after delete=4
lancedb: count before=5, after upsert=5 (should be unchanged), after delete=4


## 7. Persistence across a fresh client — real restart, not a mock

In [8]:
persist_dir = tempfile.mkdtemp(prefix="module10-persist-")

store1 = ChromaVectorStore("persist-demo", path=f"{persist_dir}/chroma")
await store1.add("d1", "persisted document", await embedder.embed_query("persisted document"))
print("store1 count:", await store1.count())

# A genuinely new client instance, not the same Python object.
store2 = ChromaVectorStore("persist-demo", path=f"{persist_dir}/chroma")
print("store2 (fresh client, same path) count:", await store2.count())


store1 count: 1
store2 (fresh client, same path) count: 1


## 8. Retrieval latency and recall@k across backends — Labs 5-6

In [9]:
from benchmark_and_evaluate import results_to_markdown_table, run_lab as run_benchmark_lab

benchmark_results = await run_benchmark_lab(tmp_dir)
print(results_to_markdown_table(benchmark_results))


# Labs 5-6 — retrieval latency and recall@k across backends

(corpus size: 5, eval cases: 3)

| Store | Mean latency (s) | Recall@k | Precision@k | MRR | nDCG@k |
|---|---:|---:|---:|---:|---:|
| numpy | 0.000017 | 1.00 | 0.44 | 0.83 | 0.88 |
| chroma | 0.000338 | 1.00 | 0.44 | 0.83 | 0.88 |
| lancedb | 0.001682 | 1.00 | 0.44 | 0.83 | 0.88 |



## 9. Cleanup

Removing the temporary on-disk stores created by this notebook.

In [10]:
import shutil

shutil.rmtree(tmp_dir, ignore_errors=True)
shutil.rmtree(persist_dir, ignore_errors=True)
print("cleaned up")


cleaned up
